# Limpieza y transformación del Dataset

In [52]:
import pandas as pd
# Cargamos el dataset
df = pd.read_csv('../../data/dataset_original.csv', encoding='ISO-8859-1')
# Mostramos las primeras filas
df.head(4)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [ ]:
df = df[df['Country'] == 'United Kingdom']
df.head()

- Filtramos líneas con precios que anulan las ventas


In [53]:
# Las facturas que tienen UnitPrice a 0, no aportan nada.
df = df[df['UnitPrice'] != 0]
df[df['UnitPrice']==0]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country


- Filtramos las líneas con precios unitarios negativos

In [54]:
mask = df[['UnitPrice']] < 0
df = df[~mask.any(axis=1)]

print(f"Numero de 'UnitPrice' negativos antes: {mask.sum()['UnitPrice']}")
print(f"Numero de 'UnitPrice' negativos ahora: {(df[['UnitPrice']] < 0).sum()['UnitPrice']}")

Numero de 'UnitPrice' negativos antes: 2
Numero de 'UnitPrice' negativos ahora: 0


- Eliminamos las filas duplicadas

In [55]:
print(f"Numero de duplicados antes: {df.duplicated().sum()}")

df = df.drop_duplicates()

print(f"Numero de duplicados después: {df.duplicated().sum()}")

Numero de duplicados antes: 5263
Numero de duplicados después: 0


- Eliminamos los valores nulos de CustomerID

In [56]:
print(f"Número de 'CustomerID' nulos antes: {df['CustomerID'].isnull().sum()}")

# agrupamos por InvoiceNo , y usamos ffill y bfill para rellenar, los valores nulls que no se rellenen les da un valor nuevo unico
df = df[df['CustomerID'].notnull()]

print(f"Número de 'CustomerID' nulos ahora: {df['CustomerID'].isnull().sum()}")


Número de 'CustomerID' nulos antes: 132565
Número de 'CustomerID' nulos ahora: 0


- Rellenamos los valores nulos de Description

In [57]:
# tratamiento Description null
print(f"Número de 'Description' nulos antes: {df['Description'].isnull().sum()}")

# agrupamos por stockcode, y usamos ffill para rellenar con el anterior valor dentro del grupo, para los primer valor en null tambien uso bfill
df['Description'] = df.groupby(['StockCode'])['Description'].transform(lambda group: group.ffill().bfill())
df['Description'] = df['Description'].fillna('No description')

print(f"Número de 'Description' nulos ahora: {df['Description'].isnull().sum()}")

Número de 'Description' nulos antes: 0
Número de 'Description' nulos ahora: 0


- Detectamos y eliminamos los "outliers"

In [58]:
# Para los outliers, usamos el rango intercuartil (IQR)
total_sales = df['Quantity'] * df['UnitPrice']

# Para los outliers, usamos el rango intercuartil (IQR)
q1 = total_sales.quantile(0.25)
q3 = total_sales.quantile(0.75)
iqr = q3 - q1

# Ajuste de rango para 90%//95%
range=2.5
lower_limit = q1 - (range * iqr)
higher_limit = q3 + (range * iqr)

outliers_condition = (total_sales >= lower_limit) & (total_sales <= higher_limit)
filtered_df = df[outliers_condition]

# porcentaje filtarado
print(f"Porcentaje de datos retenidos: {len(filtered_df) / len(df) * 100:.2f}%")

df = filtered_df
df

Porcentaje de datos retenidos: 93.78%


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,12/9/2011 12:50,0.85,12680.0,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,12/9/2011 12:50,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,12/9/2011 12:50,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,12/9/2011 12:50,4.15,12680.0,France


# Transformación

- Calculamos las ventas totales para cada conjunto

In [59]:
df['TotalSales'] = df['Quantity'] * df['UnitPrice']

- Descomponemos las fechas en varias columnas para enriquecer los datasets

In [60]:
def separate_date(df: pd.DataFrame):
  df['Year'] = df['InvoiceDate'].dt.year
  df['Month'] = df['InvoiceDate'].dt.month
  df['Day'] = df['InvoiceDate'].dt.day
  df['DayOfWeek'] = df['InvoiceDate'].dt.dayofweek
  df['Quarter'] = df['Month'].apply(
    lambda month: 
      1 if month in [1, 2, 3] else
      2 if month in [4, 5, 6] else
      3 if month in [7, 8, 9] else 
      4 # if no one chosen
  )
  df['Season'] = df['Month'].apply(
    lambda x: 
      'Winter' if x in [12, 1, 2] else
      'Spring' if x in [3, 4, 5] else
      'Summer' if x in [6, 7, 8] else 
      'Fall' # if no one chosen
  )
  return df


df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df = separate_date(df)
df

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalSales,Year,Month,Day,DayOfWeek,Quarter,Season
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,2010,12,1,2,4,Winter
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010,12,1,2,4,Winter
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,2010,12,1,2,4,Winter
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010,12,1,2,4,Winter
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010,12,1,2,4,Winter
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680.0,France,10.20,2011,12,9,4,4,Winter
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France,12.60,2011,12,9,4,4,Winter
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France,16.60,2011,12,9,4,4,Winter
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France,16.60,2011,12,9,4,4,Winter


- Guardamos el dataset de entrenamiento

In [62]:
df.to_csv('../../data/dataset_for_clustering.csv', index=False)